# Analisis Tren Iklim — Studi Kasus: Yogyakarta
### Mann-Kendall Test & Sen's Slope
**Studi Kasus:** Suhu dan Curah Hujan Tahunan Kota Yogyakarta  
**Periode:** 1990–2020 (31 tahun)

---
**Alur:**
```
1. Generate demo data (deret waktu tahunan)
2. Visualisasi awal — plot time series
3. Implementasi Mann-Kendall Test
4. Implementasi Sen's Slope
5. Analisis tren multi-variabel
6. Tabel ringkasan hasil
```

---
**Mengapa Mann-Kendall?**
- Non-parametrik → tidak perlu asumsi distribusi normal
- Cocok untuk data iklim yang sering skewed
- Sudah jadi standar WMO untuk analisis tren iklim

## 0. Import Library

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
np.random.seed(42)

print('✅ Library berhasil diimport')

---
## 1. Generate Demo Data

> **Catatan:** Dalam penelitian nyata, data ini diganti dengan:
> - Data observasi jangka panjang (≥ 30 tahun) dari stasiun BMKG
> - Atau data gridded: ERA5, CHIRPS
> - Format: data **tahunan** atau **musiman** (agregasi dari data harian)

In [ ]:
# ============================================================
# GENERATE DATA TAHUNAN YOGYAKARTA (1990–2020)
# ============================================================
years = np.arange(1990, 2021)
n = len(years)
frac = (years - 1990) / 30  # 0 → 1

# Suhu rata-rata tahunan (tren +0.25°C/dekade)
suhu = 27.3 + 0.025 * (years - 1990) + np.random.normal(0, 0.35, n)

# Curah hujan tahunan (tren lemah +20 mm/dekade, variabilitas tinggi)
hujan = 2100 + 2.0 * (years - 1990) + np.random.normal(0, 180, n)

# Hari Hujan per tahun (tren lemah)
hari_hujan = 145 + 0.5 * (years - 1990) + np.random.normal(0, 12, n)

# Suhu maksimum (tren lebih kuat +0.35°C/dekade)
suhu_max = 32.5 + 0.035 * (years - 1990) + np.random.normal(0, 0.4, n)

df = pd.DataFrame({
    'year'      : years,
    'suhu'      : suhu,
    'suhu_max'  : suhu_max,
    'hujan'     : hujan,
    'hari_hujan': hari_hujan,
})

print('📊 Data tahunan Yogyakarta (1990–2020):')
print(df.describe().round(2))

---
## 2. Visualisasi Awal

In [ ]:
# ============================================================
# PLOT TIME SERIES SEMUA VARIABEL
# ============================================================
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes = axes.flatten()

var_info = [
    ('suhu',       'Suhu Rata-rata Tahunan (°C)',       'tomato'),
    ('suhu_max',   'Suhu Maksimum Rata-rata Tahunan (°C)', 'darkorange'),
    ('hujan',      'Curah Hujan Tahunan (mm/tahun)',    'steelblue'),
    ('hari_hujan', 'Jumlah Hari Hujan per Tahun (hari)','teal'),
]

for ax, (col, title, color) in zip(axes, var_info):
    ax.plot(df['year'], df[col], 'o-', lw=1.5, ms=5, color=color, alpha=0.8)
    ax.set_title(title, fontsize=11)
    ax.set_xlabel('Tahun')

plt.suptitle('Deret Waktu Iklim Tahunan — Yogyakarta (1990–2020)', fontsize=14)
plt.tight_layout()
plt.savefig('plot_timeseries.png', dpi=120, bbox_inches='tight')
plt.show()
print('✅ Plot time series tersimpan')

---
## 3. Implementasi Mann-Kendall Test

**Prinsip:**
- Bandingkan semua pasangan data: apakah data berikutnya lebih besar (konkordant) atau lebih kecil (diskordant)?
- Statistik **S** = Σ sign(xⱼ - xᵢ) untuk semua pasangan i < j
- S > 0 → tren naik, S < 0 → tren turun
- Konversi ke **Z** → hitung **p-value**

In [ ]:
# ============================================================
# IMPLEMENTASI MANN-KENDALL TEST
# ============================================================
def mann_kendall(x):
    """
    Uji Mann-Kendall untuk deteksi tren monoton.
    
    Parameter
    ----------
    x : array-like, deret waktu
    
    Returns
    -------
    dict dengan kunci:
        S       : statistik S (jumlah konkordant - diskordant)
        Z       : statistik Z (ternormalisasi)
        p_value : p-value (two-tailed)
        tren    : 'naik', 'turun', atau 'tidak ada tren'
        signif  : True jika p < 0.05
    """
    x = np.array(x, dtype=float)
    n = len(x)
    
    # Hitung S
    S = 0
    for i in range(n - 1):
        for j in range(i + 1, n):
            S += np.sign(x[j] - x[i])
    
    # Variance of S (tanpa ties)
    var_S = n * (n - 1) * (2 * n + 5) / 18
    
    # Koreksi Z
    if S > 0:
        Z = (S - 1) / np.sqrt(var_S)
    elif S < 0:
        Z = (S + 1) / np.sqrt(var_S)
    else:
        Z = 0.0
    
    # p-value (two-tailed)
    p_value = 2 * (1 - stats.norm.cdf(abs(Z)))
    
    # Interpretasi
    if p_value < 0.05:
        tren = 'naik' if S > 0 else 'turun'
        signif = True
    else:
        tren = 'tidak ada tren'
        signif = False
    
    return {'S': int(S), 'Z': round(Z, 4),
            'p_value': round(p_value, 4),
            'tren': tren, 'signif': signif}

# --- Test pada satu variabel dulu ---
hasil_suhu = mann_kendall(df['suhu'])
print('🔍 Hasil Mann-Kendall untuk Suhu Rata-rata:')
for k, v in hasil_suhu.items():
    print(f'   {k:<10}: {v}')

---
## 4. Implementasi Sen's Slope

**Prinsip:**
- Hitung kemiringan untuk setiap pasangan titik: βᵢⱼ = (xⱼ - xᵢ) / (j - i)
- Sen's Slope = **median** dari semua βᵢⱼ
- Lebih robust dibanding regresi biasa karena tidak terpengaruh outlier

In [ ]:
# ============================================================
# IMPLEMENTASI SEN'S SLOPE
# ============================================================
def sens_slope(x):
    """
    Estimasi tren Sen's Slope.
    
    Parameter
    ----------
    x : array-like, deret waktu
    
    Returns
    -------
    slope    : besar tren per langkah waktu (unit/tahun)
    intercept: intercept garis tren
    """
    x = np.array(x, dtype=float)
    n = len(x)
    
    # Hitung semua kemiringan pasangan
    slopes = []
    for i in range(n - 1):
        for j in range(i + 1, n):
            slopes.append((x[j] - x[i]) / (j - i))
    
    slope = np.median(slopes)
    intercept = np.median(x) - slope * np.median(np.arange(n))
    return slope, intercept

# --- Test pada suhu ---
slope_suhu, intercept_suhu = sens_slope(df['suhu'].values)
print(f"Sen's Slope suhu: {slope_suhu:+.4f} °C/tahun")
print(f"                  {slope_suhu*10:+.4f} °C/dekade")
print(f'Intercept        : {intercept_suhu:.4f}')

---
## 5. Analisis Tren Multi-Variabel

In [ ]:
# ============================================================
# PLOT TREN SEMUA VARIABEL
# ============================================================
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
axes = axes.flatten()

var_info = [
    ('suhu',       'Suhu Rata-rata Tahunan (°C)',          'tomato',    '°C/dekade'),
    ('suhu_max',   'Suhu Maksimum Rata-rata Tahunan (°C)', 'darkorange','°C/dekade'),
    ('hujan',      'Curah Hujan Tahunan (mm/tahun)',       'steelblue', 'mm/dekade'),
    ('hari_hujan', 'Jumlah Hari Hujan per Tahun (hari)',   'teal',      'hari/dekade'),
]

for ax, (col, title, color, unit) in zip(axes, var_info):
    y = df[col].values
    x_idx = np.arange(len(y))
    
    # Data
    ax.plot(df['year'], y, 'o', ms=5, color=color, alpha=0.7, label='Data')
    ax.plot(df['year'], y, '-', lw=0.8, color=color, alpha=0.4)
    
    # Sen's Slope
    slope, intercept = sens_slope(y)
    trend_line = intercept + slope * x_idx
    mk = mann_kendall(y)
    
    lbl = (f"Sen's slope: {slope*10:+.3f} {unit}\n"
           f"p-value: {mk['p_value']:.3f} "
           f"({'signifikan ✓' if mk['signif'] else 'tidak signifikan'})")
    ax.plot(df['year'], trend_line, '--', lw=2.5, color='black', label=lbl)
    
    ax.set_title(title, fontsize=11)
    ax.set_xlabel('Tahun')
    ax.legend(fontsize=8.5, loc='best')

plt.suptitle("Analisis Tren Mann-Kendall & Sen's Slope — Yogyakarta", fontsize=14)
plt.tight_layout()
plt.savefig('plot_tren.png', dpi=120, bbox_inches='tight')
plt.show()
print('✅ Plot tren tersimpan')

---
## 6. Tabel Ringkasan Hasil

In [ ]:
# ============================================================
# TABEL RINGKASAN SEMUA VARIABEL
# ============================================================
var_labels = {
    'suhu'       : ('Suhu Rata-rata', '°C'),
    'suhu_max'   : ('Suhu Maksimum',  '°C'),
    'hujan'      : ('Curah Hujan',    'mm'),
    'hari_hujan' : ('Hari Hujan',     'hari'),
}

rows = []
for col, (label, unit) in var_labels.items():
    mk = mann_kendall(df[col].values)
    slope, _ = sens_slope(df[col].values)
    rows.append({
        'Variabel'           : label,
        'Statistik S'        : mk['S'],
        'Z'                  : mk['Z'],
        'p-value'            : mk['p_value'],
        'Signifikan (α=0.05)': '✓ Ya' if mk['signif'] else '✗ Tidak',
        'Tren'               : mk['tren'],
        f"Sen's Slope ({unit}/dekade)": round(slope * 10, 4),
    })

tabel = pd.DataFrame(rows).set_index('Variabel')
print('📊 Ringkasan Analisis Tren:')
print(tabel.to_string())

tabel.to_csv('hasil_trend_analysis.csv')
print('\n✅ Hasil tersimpan: hasil_trend_analysis.csv')

In [ ]:
# ============================================================
# RINGKASAN CETAK RAPI
# ============================================================
print('=' * 60)
print('  RINGKASAN TREN IKLIM — YOGYAKARTA (1990–2020)')
print('=' * 60)
for col, (label, unit) in var_labels.items():
    mk = mann_kendall(df[col].values)
    slope, _ = sens_slope(df[col].values)
    signif_str = '✓ SIGNIFIKAN' if mk['signif'] else '✗ tidak signifikan'
    print(f'\n  {label}:')
    print(f"    Sen's slope : {slope*10:+.4f} {unit}/dekade")
    print(f'    p-value     : {mk["p_value"]:.4f}  → {signif_str}')
    print(f'    Tren        : {mk["tren"].upper()}')
print()
print('  Keterangan: uji two-tailed, level signifikansi α = 0.05')
print('=' * 60)